# Loading Data, selecting features,target, divide data into training and validation

In [1]:
import pandas as pd # a powerful Python data analysis library

#load dataset
data_path="melb_data.csv"
data=pd.read_csv(data_path)

#If a row has a null target value , you just drop that row entirely. A target row with no answer is useless for training 
# — the model can't learn from it.
data.dropna(subset=["Price"],axis=0,inplace=True) # Here dropna removes null value for a row . axis=0 means drop row only.
# subset=['Price'] says — only drop a row if this specific column is null, ignore everything else. And inplace=True means modify 
# the original DataFrame directly instead of returning a new one.

# select target
y=data.Price

# To keep things simple, we'll use only numerical predictors
melb_predictors=data.drop(["Price"],axis=1) #drop price for features 
X=melb_predictors.select_dtypes(exclude=["object"])

from sklearn.model_selection import train_test_split # a machine learning technique used to evaluate model performance by dividing 
# a dataset into two distinct subsets: a training set to teach the model and a test set to evaluate its predictions

# divide data into training and validation subsets(train_size=0.8 means 80% data for training and test_size=0.2 means 20% data for validation)
train_X,val_X,train_y,val_y=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=0) 

# Define Function to Measure Quality of Each Approach

In [2]:
from sklearn.ensemble import RandomForestRegressor # a supervised machine learning algorithm used to predict 
# continuous numerical values by combining the outputs of multiple decision trees

from sklearn.metrics import mean_absolute_error # a metric used to measure the average magnitude of errors in a set of predictions

# function for comparing different approaches
def score_dataset(train_X,val_X,train_y,val_y):
    model=RandomForestRegressor(n_estimators=10,random_state=0) # n_estimators means the numbers of trees
    model.fit(train_X,train_y)
    preds=model.predict(val_X)
    mae=mean_absolute_error(val_y,preds) # Mean Absolute Error
    return mae

# Score from Approach 1 (Drop Columns with Missing Values)

In [3]:
# Since we are working with both training and validation sets, we are careful to drop the same columns in both DataFrames.

# Get names of columns with missing values
# isnull() checks every cell and returns True if empty, False if has value:
# any() Checks if at least one True exists in that series:
cols_with_missing=[]
for col in train_X.columns:
    if train_X[col].isnull().any():
        cols_with_missing.append(col)

# Drop columns in training and validation data
reduced_train_X=train_X.drop(cols_with_missing,axis=1)
reduced_val_X=val_X.drop(cols_with_missing,axis=1)

# pass parameters to score_dataset function and calulate mean absolute error 
print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_train_X, reduced_val_X, train_y, val_y))

MAE from Approach 1 (Drop columns with missing values):
183550.22137772635


# Score from Approach 2 (Imputation)

In [4]:
from sklearn.impute import SimpleImputer #SimpleImputer replaces null values using basic, column-wise statistical strategies 
# like mean, median, or a constant.

# Imputation
my_imputer=SimpleImputer()

#Filling the null values in columns with mean 
np_arr_train_X=my_imputer.fit_transform(train_X) #looks at the training data, calculates the mean of each column

# we never fit on validation data. Validation is supposed to simulate unseen data. If we fit on it, we're cheating — that's leakage.
np_arr_val_X=my_imputer.transform(val_X) # fills missing values using those calculated means from train_X

# .fit_transform() or .transform() returns a (numpy array), not a DataFrame. let's transfrom them to DataFrame
imputed_train_X=pd.DataFrame(np_arr_train_X)
imputed_val_X=pd.DataFrame(np_arr_val_X)

# Numpy arrays have no column names. So after transforming to DataFrame, they have just raw numbers.
# it auto-assigns 0, 1, 2... as column names instead of the original names. 
# so we have to get the columns back again
imputed_train_X.columns=train_X.columns
imputed_val_X.columns=val_X.columns

# pass parameters to score_dataset function and calulate mean absolute error
print("Score from Approach 2 (Imputation)")
print(score_dataset(imputed_train_X,imputed_val_X,train_y,val_y))

Score from Approach 2 (Imputation)
178166.46269899711


# Score from Approach 3 (An Extension to Imputation)

In [6]:
# make a copy to avoid changing original data when (adding new columns)
train_X_copy=train_X.copy()
val_X_copy=val_X.copy()

# Identify which columns contain null values
missing_values_columns=[]
for col in train_X_copy.columns:
    if train_X_copy[col].isnull().any():
        missing_values_columns.append(col)

# make new columns which will store bool (True/False) means (1/0) for existing columns which contain null values
for col in missing_values_columns:
    train_X_copy[col+"_was_missing"]=train_X_copy[col].isnull()
    val_X_copy[col+"_was_missing"]=val_X_copy[col].isnull()

# Imputation
np_arr_train_X_copy=my_imputer.fit_transform(train_X_copy)
np_arr_val_X_copy=my_imputer.transform(val_X_copy)

# convert numpy array to data frame
imputed_train_X_copy=pd.DataFrame(np_arr_train_X_copy)
imputed_val_X_copy=pd.DataFrame(np_arr_val_X_copy)

# Imputation removes columns name  now add to get back
imputed_train_X_copy.columns=train_X_copy.columns
imputed_val_X_copy.columns=val_X_copy.columns

# pass parameters to score_dataset function and calulate mean absolute error
print("Score from Approach 3 (An Extension to Imputation")
print(score_dataset(imputed_train_X_copy,imputed_val_X_copy,train_y,val_y))

Score from Approach 3 (An Extension to Imputation
178927.503183954


In [7]:
train_X_copy

,Rooms,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount,Car_was_missing,BuildingArea_was_missing,YearBuilt_was_missing
12167,1,5.0,3182.0,1.0,1.0,1.0,0.0,NaN,1940.0,-37.85984,144.98670,13240.0,False,True,False
6524,2,8.0,3016.0,2.0,2.0,1.0,193.0,NaN,NaN,-37.85800,144.90050,6380.0,False,True,True
8413,3,12.6,3020.0,3.0,1.0,1.0,555.0,NaN,NaN,-37.79880,144.82200,3755.0,False,True,True
2919,3,13.0,3046.0,3.0,1.0,1.0,265.0,NaN,1995.0,-37.70830,144.91580,8870.0,False,True,False
6043,3,13.3,3020.0,3.0,1.0,2.0,673.0,673.0,1970.0,-37.76230,144.82720,4217.0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13123,3,5.2,3056.0,3.0,1.0,2.0,212.0,NaN,NaN,-37.77695,144.95785,11918.0,False,True,True
3264,3,10.5,3081.0,3.0,1.0,1.0,748.0,101.0,1950.0,-37.74160,145.04810,2947.0,False,False,False
9845,4,6.7,3058.0,4.0,2.0,2.0,441.0,255.0,2002.0,-37.73572,144.97256,11204.0,False,False,False
10799,3,12.0,3073.0,3.0,1.0,1.0,606.0,NaN,NaN,-37.72057,145.02615,21650.0,False,True,True
